# NB03 - Medallion Transformation (Bronze → Silver)

## Objective

This notebook transforms raw Bronze data into clean and standardized Silver data.

### Source

- LH_Bronze (Delta Tables)

### Destination

- LH_Silver (Delta Tables)

### Activities Performed

- Read Bronze Delta tables
- Apply business transformations
- Standardize data
- Handle data quality issues
- Write cleaned tables to the Silver Lakehouse

The Silver layer contains validated, cleaned and analytics-ready data while preserving the original raw data in the Bronze layer.

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

print("Libraries loaded successfully.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 3, Finished, Available, Finished, False)

Libraries loaded successfully.


## Define Source and Destination

### Objective

Define the Bronze and Silver tables that will participate in the Medallion transformation.

The notebook reads data from the Bronze Lakehouse and writes the transformed data into the Silver Lakehouse.

In [2]:
# ==========================================================
# Tables for Medallion Transformation
# ==========================================================

tables = [

    "DimRegion",
    "DimStore",
    "DimCustomer",
    "DimSupplier",
    "DimProduct",
    "DimEmployee",
    "DimPromotion",
    "DimDate",
    "FactSales",
    "FactInventory",
    "FactReturns"

]

print(f"Total Tables : {len(tables)}")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 4, Finished, Available, Finished, False)

Total Tables : 11


## Verify Bronze Lakehouse Access

### Objective

Before applying transformations, verify that the notebook can access the Bronze Lakehouse tables.

This confirms that the notebook has access to both the Bronze and Silver Lakehouses.

In [3]:
# ==========================================================
# Bronze Base Path
# ==========================================================

bronze_base_path = "abfss://EnterpriseRetailAnalytics@onelake.dfs.fabric.microsoft.com/LH_Bronze.Lakehouse/Tables/dbo"

# Read DimRegion from Bronze

dim_region_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/dimregion")
)

# Display the data

dim_region_df.show()

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 6, Finished, Available, Finished, False)

+---------+----------+-------+
|RegionKey|RegionName|Country|
+---------+----------+-------+
|        1|     North|  India|
|        2|     South|  India|
|        3|      East|  India|
|        4|      West|  India|
|        5|   Central|  India|
|        6|North-East|  India|
+---------+----------+-------+



## Data Quality Check

### Objective

Inspect the structure and quality of the Bronze data before transformation.

The following checks are performed:

- Number of records
- Column names and data types
- Sample data

In [4]:
# ==========================================================
# Inspect DimRegion
# ==========================================================

print("Total Records :", dim_region_df.count())

print("\nSchema")
dim_region_df.printSchema()

print("\nSample Data")
dim_region_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 8, Finished, Available, Finished, False)

Total Records : 6

Schema
root
 |-- RegionKey: long (nullable = true)
 |-- RegionName: string (nullable = true)
 |-- Country: string (nullable = true)


Sample Data
+---------+----------+-------+
|RegionKey|RegionName|Country|
+---------+----------+-------+
|1        |North     |India  |
|2        |South     |India  |
|3        |East      |India  |
|4        |West      |India  |
|5        |Central   |India  |
+---------+----------+-------+
only showing top 5 rows



## Transform DimRegion

### Objective

Apply basic data quality transformations to the `DimRegion` table.

The transformations performed are:

- Remove leading and trailing spaces
- Standardize text format
- Remove duplicate records

The transformed data will be stored in a new DataFrame before writing to the Silver Lakehouse.

In [5]:
# ==========================================================
# Transform DimRegion
# ==========================================================

dim_region_silver = (

    dim_region_df

    # Remove leading/trailing spaces
    .withColumn(
        "RegionName",
        F.trim(F.col("RegionName"))
    )

    # Convert Region Name to Proper Case
    .withColumn(
        "RegionName",
        F.initcap(F.col("RegionName"))
    )

    # Remove duplicate records
    .dropDuplicates()

)

print("Transformation completed successfully.")

dim_region_silver.show()

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 9, Finished, Available, Finished, False)

Transformation completed successfully.
+---------+----------+-------+
|RegionKey|RegionName|Country|
+---------+----------+-------+
|        6|North-east|  India|
|        3|      East|  India|
|        1|     North|  India|
|        2|     South|  India|
|        5|   Central|  India|
|        4|      West|  India|
+---------+----------+-------+



## Write DimRegion to Silver

### Objective

Write the transformed `DimRegion` DataFrame to the Silver Lakehouse as a managed Delta table.

The Silver table will replace any existing version of the table.

In [6]:
# ==========================================================
# Write DimRegion to Silver
# ==========================================================

(
    dim_region_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("DimRegion")
)

print("DimRegion successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 11, Finished, Available, Finished, False)

DimRegion successfully written to LH_Silver.


## Transform DimStore

### Objective

Read the `DimStore` table from the Bronze Lakehouse and apply basic data quality transformations before loading it into the Silver Lakehouse.

### Transformations

- Remove leading and trailing spaces from text columns.
- Standardize text values using Proper Case.
- Remove duplicate records.
- Preserve all business keys.

In [7]:
# ==========================================================
# Read DimStore from Bronze
# ==========================================================

dim_store_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/dimstore")
)

print("Bronze DimStore Loaded Successfully")

dim_store_df.show(5)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 13, Finished, Available, Finished, False)

Bronze DimStore Loaded Successfully
+--------+---------+--------------+---------+-----------+---------+---------+-----------+--------+
|StoreKey|StoreCode|     StoreName|     City|      State|RegionKey|StoreType|OpeningDate|IsActive|
+--------+---------+--------------+---------+-----------+---------+---------+-----------+--------+
|       1|  STR0001|Retail Store 1|Ahmedabad|    Gujarat|        4|   Outlet| 2027-06-10|    true|
|       2|  STR0002|Retail Store 2|Hyderabad|  Telangana|        2|     Mall| 2022-12-12|    true|
|       3|  STR0003|Retail Store 3|   Mumbai|Maharashtra|        4|   Retail| 2022-03-19|    true|
|       4|  STR0004|Retail Store 4|    Delhi|      Delhi|        1|  Express| 2024-05-04|    true|
|       5|  STR0005|Retail Store 5|Bengaluru|  Karnataka|        2|  Express| 2024-01-31|    true|
+--------+---------+--------------+---------+-----------+---------+---------+-----------+--------+
only showing top 5 rows



## Inspect DimStore

### Objective

Verify the structure of the Bronze table before applying transformations.

The following validations are performed:

- Row count
- Schema
- Sample data

In [8]:
# ==========================================================
# Inspect DimStore
# ==========================================================

print("Total Records :", dim_store_df.count())

print("\nSchema")
dim_store_df.printSchema()

print("\nSample Data")
dim_store_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 15, Finished, Available, Finished, False)

Total Records : 100

Schema
root
 |-- StoreKey: long (nullable = true)
 |-- StoreCode: string (nullable = true)
 |-- StoreName: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- RegionKey: long (nullable = true)
 |-- StoreType: string (nullable = true)
 |-- OpeningDate: date (nullable = true)
 |-- IsActive: boolean (nullable = true)


Sample Data
+--------+---------+--------------+---------+-----------+---------+---------+-----------+--------+
|StoreKey|StoreCode|StoreName     |City     |State      |RegionKey|StoreType|OpeningDate|IsActive|
+--------+---------+--------------+---------+-----------+---------+---------+-----------+--------+
|1       |STR0001  |Retail Store 1|Ahmedabad|Gujarat    |4        |Outlet   |2027-06-10 |true    |
|2       |STR0002  |Retail Store 2|Hyderabad|Telangana  |2        |Mall     |2022-12-12 |true    |
|3       |STR0003  |Retail Store 3|Mumbai   |Maharashtra|4        |Retail   |2022-03-19 |true    |
|4  

## Transform DimStore

### Objective

Apply business transformations to the DimStore table.

The following transformations are performed:

- Remove leading and trailing spaces from text columns.
- Standardize text columns using Proper Case.
- Remove duplicate records.
- Preserve business keys and data types.

In [9]:
# ==========================================================
# Transform DimStore
# ==========================================================

dim_store_silver = (

    dim_store_df

    # Store Name
    .withColumn(
        "StoreName",
        F.initcap(F.trim(F.col("StoreName")))
    )

    # City
    .withColumn(
        "City",
        F.initcap(F.trim(F.col("City")))
    )

    # State
    .withColumn(
        "State",
        F.initcap(F.trim(F.col("State")))
    )

    # Store Type
    .withColumn(
        "StoreType",
        F.initcap(F.trim(F.col("StoreType")))
    )

    # Remove duplicate rows
    .dropDuplicates()

)

print("DimStore transformed successfully.")

dim_store_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 16, Finished, Available, Finished, False)

DimStore transformed successfully.
+--------+---------+---------------+---------+-----------+---------+---------+-----------+--------+
|StoreKey|StoreCode|StoreName      |City     |State      |RegionKey|StoreType|OpeningDate|IsActive|
+--------+---------+---------------+---------+-----------+---------+---------+-----------+--------+
|48      |STR0048  |Retail Store 48|Hyderabad|Telangana  |2        |Mall     |2024-04-01 |true    |
|24      |STR0024  |Retail Store 24|Ahmedabad|Gujarat    |4        |Mall     |2027-07-13 |true    |
|77      |STR0077  |Retail Store 77|Mumbai   |Maharashtra|4        |Mall     |2025-02-24 |true    |
|16      |STR0016  |Retail Store 16|Bengaluru|Karnataka  |2        |Retail   |2022-10-19 |true    |
|50      |STR0050  |Retail Store 50|Delhi    |Delhi      |1        |Retail   |2025-11-28 |true    |
+--------+---------+---------------+---------+-----------+---------+---------+-----------+--------+
only showing top 5 rows



## Write DimStore to Silver

### Objective

Write the transformed `DimStore` table to the Silver Lakehouse as a managed Delta table.

Any existing version of the table will be replaced.

In [10]:
# ==========================================================
# Write DimStore to Silver
# ==========================================================

(
    dim_store_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("DimStore")
)

print("DimStore successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 17, Finished, Available, Finished, False)

DimStore successfully written to LH_Silver.


## Read DimCustomer

### Objective

Read the `DimCustomer` table from the Bronze Lakehouse.

This table contains customer master data and will be transformed into a clean, standardized Silver table.

In [11]:
# ==========================================================
# Read DimCustomer from Bronze
# ==========================================================

dim_customer_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/dimcustomer")
)

print("Bronze DimCustomer Loaded Successfully")

dim_customer_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 18, Finished, Available, Finished, False)

Bronze DimCustomer Loaded Successfully
+-----------+----------+----------------+------+-----------+---+--------+---------+-----------+---------+----------------------------+-------------+----------+-----------+
|CustomerKey|CustomerID|CustomerName    |Gender|DateOfBirth|Age|AgeGroup|City     |State      |RegionKey|Email                       |Phone        |JoinDate  |LoyaltyTier|
+-----------+----------+----------------+------+-----------+---+--------+---------+-----------+---------+----------------------------+-------------+----------+-----------+
|1          |CUST000001|Veer Balay      |Female|1977-01-27 |49 |45-54   |Mumbai   |Maharashtra|4        |veer.balay@example.com      |01395376724  |2022-06-05|Platinum   |
|2          |CUST000002|Yatin Yohannan  |Female|1973-06-27 |53 |45-54   |Delhi    |Delhi      |1        |yatin.yohannan@example.com  |6965328710   |2021-10-18|Gold       |
|3          |CUST000003|Michael Dutta   |Male  |2001-11-30 |24 |18-24   |Mumbai   |Maharashtra|4     

## Inspect DimCustomer

### Objective

Inspect the Bronze DimCustomer table before applying any transformations.

The inspection includes:

- Total records
- Schema
- Sample data

In [12]:
# ==========================================================
# Inspect DimCustomer
# ==========================================================

print("Total Records :", dim_customer_df.count())

print("\nSchema")
dim_customer_df.printSchema()

print("\nSample Data")
dim_customer_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 19, Finished, Available, Finished, False)

Total Records : 50000

Schema
root
 |-- CustomerKey: long (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- DateOfBirth: date (nullable = true)
 |-- Age: long (nullable = true)
 |-- AgeGroup: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- RegionKey: long (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- JoinDate: date (nullable = true)
 |-- LoyaltyTier: string (nullable = true)


Sample Data
+-----------+----------+----------------+------+-----------+---+--------+---------+-----------+---------+----------------------------+-------------+----------+-----------+
|CustomerKey|CustomerID|CustomerName    |Gender|DateOfBirth|Age|AgeGroup|City     |State      |RegionKey|Email                       |Phone        |JoinDate  |LoyaltyTier|
+-----------+----------+----------------+------+-----------+-

## Transform DimCustomer

### Objective

Apply business transformations to the `DimCustomer` table.

### Transformations Performed

- Trim leading and trailing spaces
- Standardize customer names
- Standardize gender values
- Standardize city and state names
- Convert email addresses to lowercase
- Remove duplicate records

In [13]:
# ==========================================================
# Transform DimCustomer
# ==========================================================

dim_customer_silver = (

    dim_customer_df

    # Customer Name
    .withColumn(
        "CustomerName",
        F.initcap(F.trim(F.col("CustomerName")))
    )

    # Gender
    .withColumn(
        "Gender",
        F.initcap(F.trim(F.col("Gender")))
    )

    # City
    .withColumn(
        "City",
        F.initcap(F.trim(F.col("City")))
    )

    # State
    .withColumn(
        "State",
        F.initcap(F.trim(F.col("State")))
    )

    # Email
    .withColumn(
        "Email",
        F.lower(F.trim(F.col("Email")))
    )

    # Phone
    .withColumn(
        "Phone",
        F.trim(F.col("Phone"))
    )

    # Loyalty Tier
    .withColumn(
        "LoyaltyTier",
        F.initcap(F.trim(F.col("LoyaltyTier")))
    )

    # Remove duplicate rows
    .dropDuplicates()

)

print("DimCustomer transformed successfully.")

dim_customer_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 20, Finished, Available, Finished, False)

DimCustomer transformed successfully.
+-----------+----------+----------------+------+-----------+---+--------+---------+-----------+---------+----------------------------+-------------+----------+-----------+
|CustomerKey|CustomerID|CustomerName    |Gender|DateOfBirth|Age|AgeGroup|City     |State      |RegionKey|Email                       |Phone        |JoinDate  |LoyaltyTier|
+-----------+----------+----------------+------+-----------+---+--------+---------+-----------+---------+----------------------------+-------------+----------+-----------+
|161        |CUST000161|Tejas Tiwari    |Male  |1986-08-22 |39 |35-44   |Chennai  |Tamil Nadu |2        |tejas.tiwari@example.com    |+919977008860|2022-05-25|Diamond    |
|419        |CUST000419|Rachana Mahajan |Female|1978-05-27 |48 |45-54   |Kolkata  |West Bengal|3        |rachana.mahajan@example.com |01496739385  |2025-04-15|Gold       |
|562        |CUST000562|Ira Puri        |Male  |1996-07-11 |30 |25-34   |Pune     |Maharashtra|4      

## Write DimCustomer to Silver

### Objective

Write the transformed `DimCustomer` table into the Silver Lakehouse as a managed Delta table.

The existing table, if present, will be replaced.

In [14]:
# ==========================================================
# Write DimCustomer to Silver
# ==========================================================

(
    dim_customer_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("DimCustomer")
)

print("DimCustomer successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 21, Finished, Available, Finished, False)

DimCustomer successfully written to LH_Silver.


## Read DimSupplier

### Objective

Read the `DimSupplier` table from the Bronze Lakehouse.

This table contains supplier master data and will be transformed into a clean, standardized Silver table.

In [15]:
# ==========================================================
# Read DimSupplier from Bronze
# ==========================================================

dim_supplier_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/dimsupplier")
)

print("Bronze DimSupplier Loaded Successfully")

dim_supplier_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 22, Finished, Available, Finished, False)

Bronze DimSupplier Loaded Successfully
+-----------+------------+---------------------------+---------------+---------+-----------+---------+
|SupplierKey|SupplierCode|SupplierName               |GSTIN          |City     |State      |RegionKey|
+-----------+------------+---------------------------+---------------+---------+-----------+---------+
|1          |SUP0001     |Prime Suppliers Pvt Ltd    |85AAAAA4213A8Z4|Mumbai   |Maharashtra|4        |
|2          |SUP0002     |Global Suppliers Pvt Ltd   |72AAAAA0620A8Z7|Surat    |Gujarat    |4        |
|3          |SUP0003     |National Suppliers Pvt Ltd |89AAAAA2666A0Z1|Ahmedabad|Gujarat    |4        |
|4          |SUP0004     |Smart Suppliers Pvt Ltd    |35AAAAA1986A6Z5|Mumbai   |Maharashtra|4        |
|5          |SUP0005     |Universal Suppliers Pvt Ltd|08AAAAA9857A1Z7|Mumbai   |Maharashtra|4        |
+-----------+------------+---------------------------+---------------+---------+-----------+---------+
only showing top 5 rows



## Inspect DimSupplier

### Objective

Inspect the Bronze `DimSupplier` table before applying transformations.

The inspection includes:

- Total record count
- Schema
- Sample data

In [16]:
# ==========================================================
# Inspect DimSupplier
# ==========================================================

print("Total Records :", dim_supplier_df.count())

print("\nSchema")
dim_supplier_df.printSchema()

print("\nSample Data")
dim_supplier_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 23, Finished, Available, Finished, False)

Total Records : 500

Schema
root
 |-- SupplierKey: long (nullable = true)
 |-- SupplierCode: string (nullable = true)
 |-- SupplierName: string (nullable = true)
 |-- GSTIN: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- RegionKey: long (nullable = true)


Sample Data
+-----------+------------+---------------------------+---------------+---------+-----------+---------+
|SupplierKey|SupplierCode|SupplierName               |GSTIN          |City     |State      |RegionKey|
+-----------+------------+---------------------------+---------------+---------+-----------+---------+
|1          |SUP0001     |Prime Suppliers Pvt Ltd    |85AAAAA4213A8Z4|Mumbai   |Maharashtra|4        |
|2          |SUP0002     |Global Suppliers Pvt Ltd   |72AAAAA0620A8Z7|Surat    |Gujarat    |4        |
|3          |SUP0003     |National Suppliers Pvt Ltd |89AAAAA2666A0Z1|Ahmedabad|Gujarat    |4        |
|4          |SUP0004     |Smart Suppliers Pvt Ltd    |35A

## Transform DimSupplier

### Objective

Apply business transformations to the `DimSupplier` table.

### Transformations Performed

- Remove leading and trailing spaces.
- Standardize supplier names.
- Convert GSTIN to uppercase.
- Standardize city and state names.
- Remove duplicate records.

In [17]:
# ==========================================================
# Transform DimSupplier
# ==========================================================

dim_supplier_silver = (

    dim_supplier_df

    # Supplier Name
    .withColumn(
        "SupplierName",
        F.initcap(F.trim(F.col("SupplierName")))
    )

    # GSTIN
    .withColumn(
        "GSTIN",
        F.upper(F.trim(F.col("GSTIN")))
    )

    # City
    .withColumn(
        "City",
        F.initcap(F.trim(F.col("City")))
    )

    # State
    .withColumn(
        "State",
        F.initcap(F.trim(F.col("State")))
    )

    # Remove duplicate rows
    .dropDuplicates()

)

print("DimSupplier transformed successfully.")

dim_supplier_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 24, Finished, Available, Finished, False)

DimSupplier transformed successfully.
+-----------+------------+------------------------+---------------+---------+-----------+---------+
|SupplierKey|SupplierCode|SupplierName            |GSTIN          |City     |State      |RegionKey|
+-----------+------------+------------------------+---------------+---------+-----------+---------+
|20         |SUP0020     |Vision Suppliers Pvt Ltd|66AAAAA7075A8Z4|Mumbai   |Maharashtra|4        |
|312        |SUP0312     |Modern Suppliers Pvt Ltd|66AAAAA0964A9Z3|Pune     |Maharashtra|4        |
|367        |SUP0367     |Smart Suppliers Pvt Ltd |90AAAAA8426A9Z5|Surat    |Gujarat    |4        |
|398        |SUP0398     |Smart Suppliers Pvt Ltd |71AAAAA8977A5Z3|Delhi    |Delhi      |1        |
|416        |SUP0416     |Smart Suppliers Pvt Ltd |55AAAAA6066A0Z1|Ahmedabad|Gujarat    |4        |
+-----------+------------+------------------------+---------------+---------+-----------+---------+
only showing top 5 rows



## Write DimSupplier to Silver

### Objective

Write the transformed `DimSupplier` table into the Silver Lakehouse as a managed Delta table.

The existing table, if present, will be replaced.

In [18]:
# ==========================================================
# Write DimSupplier to Silver
# ==========================================================

(
    dim_supplier_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("DimSupplier")
)

print("DimSupplier successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 25, Finished, Available, Finished, False)

DimSupplier successfully written to LH_Silver.


## Read DimProduct

### Objective

Read the `DimProduct` table from the Bronze Lakehouse.

This table contains product master data and will be transformed into a standardized Silver table for reporting and analytics.

In [19]:
# ==========================================================
# Read DimProduct from Bronze
# ==========================================================

dim_product_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/dimproduct")
)

print("Bronze DimProduct Loaded Successfully")

dim_product_df.show(5, truncate=False)    

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 26, Finished, Available, Finished, False)

Bronze DimProduct Loaded Successfully
+----------+-----------+--------------------+--------------+-----------+-------+-----------+--------+---------+--------+
|ProductKey|ProductCode|ProductName         |Category      |SubCategory|Brand  |SupplierKey|UnitCost|UnitPrice|IsActive|
+----------+-----------+--------------------+--------------+-----------+-------+-----------+--------+---------+--------+
|1         |PRD00001   |Puma Football 1     |Sports        |Football   |Puma   |202        |2460.43 |3091.93  |true    |
|2         |PRD00002   |Dell Decor 2        |Home & Kitchen|Decor      |Dell   |499        |2520.67 |3656.13  |true    |
|3         |PRD00003   |Samsung Headphones 3|Electronics   |Headphones |Samsung|215        |4251.46 |6497.76  |true    |
|4         |PRD00004   |Boat Men 4          |Fashion       |Men        |Boat   |482        |2349.09 |4030.47  |true    |
|5         |PRD00005   |Boat Comics 5       |Books         |Comics     |Boat   |38         |1093.47 |1469.18  |true

## Inspect DimProduct

### Objective

Inspect the Bronze `DimProduct` table before applying business transformations.

The inspection includes:

- Total records
- Schema
- Sample data

In [20]:
# ==========================================================
# Inspect DimProduct
# ==========================================================

print("Total Records :", dim_product_df.count())

print("\nSchema")
dim_product_df.printSchema()

print("\nSample Data")
dim_product_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 27, Finished, Available, Finished, False)

Total Records : 5000

Schema
root
 |-- ProductKey: long (nullable = true)
 |-- ProductCode: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- SupplierKey: long (nullable = true)
 |-- UnitCost: double (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- IsActive: boolean (nullable = true)


Sample Data
+----------+-----------+--------------------+--------------+-----------+-------+-----------+--------+---------+--------+
|ProductKey|ProductCode|ProductName         |Category      |SubCategory|Brand  |SupplierKey|UnitCost|UnitPrice|IsActive|
+----------+-----------+--------------------+--------------+-----------+-------+-----------+--------+---------+--------+
|1         |PRD00001   |Puma Football 1     |Sports        |Football   |Puma   |202        |2460.43 |3091.93  |true    |
|2         |PRD00002   |Dell Decor 2        |Home & Kitchen

## Transform DimProduct

### Objective

Apply business transformations to the `DimProduct` table.

### Transformations Performed

- Remove leading and trailing spaces.
- Standardize product-related text columns.
- Remove duplicate records.
- Preserve business identifiers, foreign keys, and numeric values.

In [21]:
# ==========================================================
# Transform DimProduct
# ==========================================================

dim_product_silver = (

    dim_product_df

    # Product Name
    .withColumn(
        "ProductName",
        F.initcap(F.trim(F.col("ProductName")))
    )

    # Category
    .withColumn(
        "Category",
        F.initcap(F.trim(F.col("Category")))
    )

    # SubCategory
    .withColumn(
        "SubCategory",
        F.initcap(F.trim(F.col("SubCategory")))
    )

    # Brand
    .withColumn(
        "Brand",
        F.initcap(F.trim(F.col("Brand")))
    )

    # Remove duplicate rows
    .dropDuplicates()

)

print("DimProduct transformed successfully.")

dim_product_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 28, Finished, Available, Finished, False)

DimProduct transformed successfully.
+----------+-----------+-------------------+-----------+-----------+-------+-----------+--------+---------+--------+
|ProductKey|ProductCode|ProductName        |Category   |SubCategory|Brand  |SupplierKey|UnitCost|UnitPrice|IsActive|
+----------+-----------+-------------------+-----------+-----------+-------+-----------+--------+---------+--------+
|65        |PRD00065   |Dell Kids 65       |Fashion    |Kids       |Dell   |68         |3139.1  |4554.32  |true    |
|384       |PRD00384   |Samsung Men 384    |Fashion    |Men        |Samsung|34         |995.7   |1575.53  |true    |
|408       |PRD00408   |Boat Laptop 408    |Electronics|Laptop     |Boat   |352        |3456.98 |4954.99  |true    |
|806       |PRD00806   |Lg Education 806   |Books      |Education  |Lg     |232        |866.98  |1287.57  |true    |
|1056      |PRD01056   |Boat Beverages 1056|Groceries  |Beverages  |Boat   |95         |1212.74 |1690.54  |true    |
+----------+-----------+---

## Write DimProduct to Silver

### Objective

Write the transformed `DimProduct` table into the Silver Lakehouse as a managed Delta table.

If the table already exists, it will be overwritten.

In [22]:
# ==========================================================
# Write DimProduct to Silver
# ==========================================================

(
    dim_product_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("DimProduct")
)

print("DimProduct successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 29, Finished, Available, Finished, False)

DimProduct successfully written to LH_Silver.


## Read DimEmployee

### Objective

Read the `DimEmployee` table from the Bronze Lakehouse.

This table contains employee master data and will be transformed into a standardized Silver table.

In [23]:
# ==========================================================
# Read DimEmployee from Bronze
# ==========================================================

dim_employee_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/dimemployee")
)

print("Bronze DimEmployee Loaded Successfully")

dim_employee_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 30, Finished, Available, Finished, False)

Bronze DimEmployee Loaded Successfully
+-----------+----------+-----------------+----------------+----------------+--------+----------+--------+
|EmployeeKey|EmployeeID|EmployeeName     |Department      |Designation     |StoreKey|HireDate  |Salary  |
+-----------+----------+-----------------+----------------+----------------+--------+----------+--------+
|1          |EMP00001  |Ishanvi Rajagopal|Sales           |Manager         |60      |2026-04-04|70853.33|
|2          |EMP00002  |Waida Luthra     |Inventory       |Senior Executive|95      |2022-03-27|36417.07|
|3          |EMP00003  |Brijesh Amble    |Customer Service|Senior Executive|78      |2021-05-17|32234.69|
|4          |EMP00004  |Warhi Vaidya     |Operations      |Supervisor      |39      |2022-10-10|46704.0 |
|5          |EMP00005  |Tanmayi Panchal  |Customer Service|Executive       |100     |2026-03-07|27371.91|
+-----------+----------+-----------------+----------------+----------------+--------+----------+--------+
only sh

## Inspect DimEmployee

### Objective

Inspect the Bronze `DimEmployee` table before applying business transformations.

The inspection includes:

- Total records
- Schema
- Sample data

In [24]:
# ==========================================================
# Inspect DimEmployee
# ==========================================================

print("Total Records :", dim_employee_df.count())

print("\nSchema")
dim_employee_df.printSchema()

print("\nSample Data")
dim_employee_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 31, Finished, Available, Finished, False)

Total Records : 500

Schema
root
 |-- EmployeeKey: long (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- EmployeeName: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Designation: string (nullable = true)
 |-- StoreKey: long (nullable = true)
 |-- HireDate: date (nullable = true)
 |-- Salary: double (nullable = true)


Sample Data
+-----------+----------+-----------------+----------------+----------------+--------+----------+--------+
|EmployeeKey|EmployeeID|EmployeeName     |Department      |Designation     |StoreKey|HireDate  |Salary  |
+-----------+----------+-----------------+----------------+----------------+--------+----------+--------+
|1          |EMP00001  |Ishanvi Rajagopal|Sales           |Manager         |60      |2026-04-04|70853.33|
|2          |EMP00002  |Waida Luthra     |Inventory       |Senior Executive|95      |2022-03-27|36417.07|
|3          |EMP00003  |Brijesh Amble    |Customer Service|Senior Executive|78      |2021-05-17|3

## Transform DimEmployee

### Objective

Apply business transformations to the `DimEmployee` table.

### Transformations Performed

- Remove leading and trailing spaces.
- Standardize employee names.
- Standardize department names.
- Standardize designation names.
- Remove duplicate records.
- Preserve business identifiers, foreign keys, and numeric values.

In [25]:
# ==========================================================
# Transform DimEmployee
# ==========================================================

dim_employee_silver = (

    dim_employee_df

    # Employee Name
    .withColumn(
        "EmployeeName",
        F.initcap(F.trim(F.col("EmployeeName")))
    )

    # Department
    .withColumn(
        "Department",
        F.initcap(F.trim(F.col("Department")))
    )

    # Designation
    .withColumn(
        "Designation",
        F.initcap(F.trim(F.col("Designation")))
    )

    # Remove duplicate rows
    .dropDuplicates()

)

print("DimEmployee transformed successfully.")

dim_employee_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 32, Finished, Available, Finished, False)

DimEmployee transformed successfully.
+-----------+----------+-----------------+----------+----------------+--------+----------+--------+
|EmployeeKey|EmployeeID|EmployeeName     |Department|Designation     |StoreKey|HireDate  |Salary  |
+-----------+----------+-----------------+----------+----------------+--------+----------+--------+
|95         |EMP00095  |Yoshita Boase    |Finance   |Senior Executive|55      |2025-04-12|45907.14|
|179        |EMP00179  |Dayita Dara      |Inventory |Senior Executive|35      |2021-02-11|40234.72|
|1          |EMP00001  |Ishanvi Rajagopal|Sales     |Manager         |60      |2026-04-04|70853.33|
|249        |EMP00249  |Arunima Kalita   |Inventory |Executive       |26      |2025-07-18|28482.2 |
|334        |EMP00334  |Christopher Bajwa|Sales     |Executive       |69      |2024-07-25|28156.95|
+-----------+----------+-----------------+----------+----------------+--------+----------+--------+
only showing top 5 rows



## Write DimEmployee to Silver

### Objective

Write the transformed `DimEmployee` table into the Silver Lakehouse as a managed Delta table.

If the table already exists, it will be overwritten.

In [26]:
# ==========================================================
# Write DimEmployee to Silver
# ==========================================================

(
    dim_employee_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("DimEmployee")
)

print("DimEmployee successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 33, Finished, Available, Finished, False)

DimEmployee successfully written to LH_Silver.


## Read DimPromotion

### Objective

Read the `DimPromotion` table from the Bronze Lakehouse.

This table contains promotion master data and will be transformed into a standardized Silver table.

In [27]:
# ==========================================================
# Read DimPromotion from Bronze
# ==========================================================

dim_promotion_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/dimpromotion")
)

print("Bronze DimPromotion Loaded Successfully")

dim_promotion_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 34, Finished, Available, Finished, False)

Bronze DimPromotion Loaded Successfully
+------------+--------------+-------------+---------------+----------+----------+
|PromotionKey|PromotionName |PromotionType|DiscountPercent|StartDate |EndDate   |
+------------+--------------+-------------+---------------+----------+----------+
|1           |Flash Sale    |Cashback     |10             |2024-07-01|2024-07-11|
|2           |Diwali Sale   |Flat Discount|5              |2027-12-07|2027-12-22|
|3           |Diwali Sale   |Flat Discount|15             |2025-11-06|2025-11-15|
|4           |Festival Offer|Percentage   |10             |2027-12-03|2027-12-11|
|5           |Diwali Sale   |Percentage   |15             |2027-12-20|2028-01-09|
+------------+--------------+-------------+---------------+----------+----------+
only showing top 5 rows



## Inspect DimPromotion

### Objective

Inspect the Bronze `DimPromotion` table before applying business transformations.

The inspection includes:

- Total records
- Schema
- Sample data

In [28]:
# ==========================================================
# Inspect DimPromotion
# ==========================================================

print("Total Records :", dim_promotion_df.count())

print("\nSchema")
dim_promotion_df.printSchema()

print("\nSample Data")
dim_promotion_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 35, Finished, Available, Finished, False)

Total Records : 100

Schema
root
 |-- PromotionKey: long (nullable = true)
 |-- PromotionName: string (nullable = true)
 |-- PromotionType: string (nullable = true)
 |-- DiscountPercent: long (nullable = true)
 |-- StartDate: date (nullable = true)
 |-- EndDate: date (nullable = true)


Sample Data
+------------+--------------+-------------+---------------+----------+----------+
|PromotionKey|PromotionName |PromotionType|DiscountPercent|StartDate |EndDate   |
+------------+--------------+-------------+---------------+----------+----------+
|1           |Flash Sale    |Cashback     |10             |2024-07-01|2024-07-11|
|2           |Diwali Sale   |Flat Discount|5              |2027-12-07|2027-12-22|
|3           |Diwali Sale   |Flat Discount|15             |2025-11-06|2025-11-15|
|4           |Festival Offer|Percentage   |10             |2027-12-03|2027-12-11|
|5           |Diwali Sale   |Percentage   |15             |2027-12-20|2028-01-09|
+------------+--------------+-------------+-

## Transform DimPromotion

### Objective

Apply business transformations to the `DimPromotion` table.

### Transformations Performed

- Remove leading and trailing spaces.
- Standardize promotion names.
- Standardize promotion types.
- Remove duplicate records.
- Preserve dates and discount values.

In [29]:
# ==========================================================
# Transform DimPromotion
# ==========================================================

dim_promotion_silver = (

    dim_promotion_df

    # Promotion Name
    .withColumn(
        "PromotionName",
        F.initcap(F.trim(F.col("PromotionName")))
    )

    # Promotion Type
    .withColumn(
        "PromotionType",
        F.initcap(F.trim(F.col("PromotionType")))
    )

    # Remove duplicate rows
    .dropDuplicates()

)

print("DimPromotion transformed successfully.")

dim_promotion_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 36, Finished, Available, Finished, False)

DimPromotion transformed successfully.
+------------+--------------+-------------+---------------+----------+----------+
|PromotionKey|PromotionName |PromotionType|DiscountPercent|StartDate |EndDate   |
+------------+--------------+-------------+---------------+----------+----------+
|15          |Clearance Sale|Percentage   |15             |2023-08-19|2023-08-28|
|94          |Clearance Sale|Flat Discount|5              |2024-06-11|2024-06-26|
|51          |Clearance Sale|Flat Discount|10             |2023-04-02|2023-04-22|
|80          |Weekend Sale  |Percentage   |15             |2023-08-13|2023-08-26|
|21          |Weekend Sale  |Cashback     |15             |2024-03-13|2024-03-26|
+------------+--------------+-------------+---------------+----------+----------+
only showing top 5 rows



## Write DimPromotion to Silver

### Objective

Write the transformed `DimPromotion` table into the Silver Lakehouse as a managed Delta table.

If the table already exists, it will be overwritten.

In [30]:
# ==========================================================
# Write DimPromotion to Silver
# ==========================================================

(
    dim_promotion_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("DimPromotion")
)

print("DimPromotion successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 37, Finished, Available, Finished, False)

DimPromotion successfully written to LH_Silver.


## Read DimDate

### Objective

Read the `DimDate` table from the Bronze Lakehouse.

This table contains the calendar dimension used for time intelligence and reporting.

In [31]:
# ==========================================================
# Read DimDate from Bronze
# ==========================================================

dim_date_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/dimdate")
)

print("Bronze DimDate Loaded Successfully")

dim_date_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 38, Finished, Available, Finished, False)

Bronze DimDate Loaded Successfully
+--------+-------------------+----+-------+-----------+---------+---------+---------------+----------+---+---------+---------+----------+
|DateKey |Date               |Year|Quarter|MonthNumber|MonthName|YearMonth|YearMonthNumber|WeekNumber|Day|DayName  |IsWeekend|FiscalYear|
+--------+-------------------+----+-------+-----------+---------+---------+---------------+----------+---+---------+---------+----------+
|20220101|2022-01-01 00:00:00|2022|Q1     |1          |January  |2022-01  |202201         |52        |1  |Saturday |true     |2021      |
|20220102|2022-01-02 00:00:00|2022|Q1     |1          |January  |2022-01  |202201         |52        |2  |Sunday   |true     |2021      |
|20220103|2022-01-03 00:00:00|2022|Q1     |1          |January  |2022-01  |202201         |1         |3  |Monday   |false    |2021      |
|20220104|2022-01-04 00:00:00|2022|Q1     |1          |January  |2022-01  |202201         |1         |4  |Tuesday  |false    |2021      |

## Inspect DimDate

### Objective

Inspect the Bronze `DimDate` table before applying business transformations.

The inspection includes:

- Total records
- Schema
- Sample data

In [32]:
# ==========================================================
# Inspect DimDate
# ==========================================================

print("Total Records :", dim_date_df.count())

print("\nSchema")
dim_date_df.printSchema()

print("\nSample Data")
dim_date_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 39, Finished, Available, Finished, False)

Total Records : 2191

Schema
root
 |-- DateKey: long (nullable = true)
 |-- Date: timestamp_ntz (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Quarter: string (nullable = true)
 |-- MonthNumber: integer (nullable = true)
 |-- MonthName: string (nullable = true)
 |-- YearMonth: string (nullable = true)
 |-- YearMonthNumber: integer (nullable = true)
 |-- WeekNumber: long (nullable = true)
 |-- Day: integer (nullable = true)
 |-- DayName: string (nullable = true)
 |-- IsWeekend: boolean (nullable = true)
 |-- FiscalYear: integer (nullable = true)


Sample Data
+--------+-------------------+----+-------+-----------+---------+---------+---------------+----------+---+---------+---------+----------+
|DateKey |Date               |Year|Quarter|MonthNumber|MonthName|YearMonth|YearMonthNumber|WeekNumber|Day|DayName  |IsWeekend|FiscalYear|
+--------+-------------------+----+-------+-----------+---------+---------+---------------+----------+---+---------+---------+----------+
|2022010

## Transform DimDate

### Objective

Apply business transformations to the `DimDate` table.

### Transformations Performed

- Standardize Quarter values.
- Standardize Month names.
- Standardize Day names.
- Remove duplicate records.

In [33]:
# ==========================================================
# Transform DimDate
# ==========================================================

dim_date_silver = (

    dim_date_df

    # Quarter
    .withColumn(
        "Quarter",
        F.upper(F.trim(F.col("Quarter")))
    )

    # Month Name
    .withColumn(
        "MonthName",
        F.initcap(F.trim(F.col("MonthName")))
    )

    # Day Name
    .withColumn(
        "DayName",
        F.initcap(F.trim(F.col("DayName")))
    )

    # Remove duplicate rows
    .dropDuplicates()

)

print("DimDate transformed successfully.")

dim_date_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 40, Finished, Available, Finished, False)

DimDate transformed successfully.
+--------+-------------------+----+-------+-----------+---------+---------+---------------+----------+---+---------+---------+----------+
|DateKey |Date               |Year|Quarter|MonthNumber|MonthName|YearMonth|YearMonthNumber|WeekNumber|Day|DayName  |IsWeekend|FiscalYear|
+--------+-------------------+----+-------+-----------+---------+---------+---------------+----------+---+---------+---------+----------+
|20220227|2022-02-27 00:00:00|2022|Q1     |2          |February |2022-02  |202202         |8         |27 |Sunday   |true     |2021      |
|20220618|2022-06-18 00:00:00|2022|Q2     |6          |June     |2022-06  |202206         |24        |18 |Saturday |true     |2022      |
|20220908|2022-09-08 00:00:00|2022|Q3     |9          |September|2022-09  |202209         |36        |8  |Thursday |false    |2022      |
|20230503|2023-05-03 00:00:00|2023|Q2     |5          |May      |2023-05  |202305         |18        |3  |Wednesday|false    |2023      |


## Write DimDate to Silver

### Objective

Write the transformed `DimDate` table into the Silver Lakehouse as a managed Delta table.

If the table already exists, it will be overwritten.

In [34]:
# ==========================================================
# Write DimDate to Silver
# ==========================================================

(
    dim_date_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("DimDate")
)

print("DimDate successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 41, Finished, Available, Finished, False)

AnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 10719fe1-a945-434c-a4ec-284e84adf60b).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- DateKey: long (nullable = true)
-- Date: timestamp_ntz (nullable = true)
-- Year: integer (nullable = true)
-- Quarter: string (nullable = true)
-- MonthNumber: integer (nullable = true)
-- MonthName: string (nullable = true)
-- WeekNumber: long (nullable = true)
-- Day: integer (nullable = true)
-- DayName: string (nullable = true)
-- IsWeekend: boolean (nullable = true)
-- FiscalYear: integer (nullable = true)


Data schema:
root
-- DateKey: long (nullable = true)
-- Date: timestamp_ntz (nullable = true)
-- Year: integer 

## UPDATED CODE

In [35]:
# ==========================================================
# Write DimDate to Silver
# ==========================================================

(
    dim_date_silver.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("DimDate")
)

print("DimDate successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 42, Finished, Available, Finished, False)

DimDate successfully written to LH_Silver.


## Read FactSales

### Objective

Read the `FactSales` table from the Bronze Lakehouse.

This table contains retail sales transactions and will be validated and transformed before loading into the Silver Lakehouse.

In [36]:
# ==========================================================
# Read FactSales from Bronze
# ==========================================================

fact_sales_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/factsales")
)

print("Bronze FactSales Loaded Successfully")

fact_sales_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 43, Finished, Available, Finished, False)

Bronze FactSales Loaded Successfully
+--------+------------+-----------+----------+--------+-----------+------------+--------+---------+--------------+-----------+--------+
|SalesKey|SalesDateKey|CustomerKey|ProductKey|StoreKey|EmployeeKey|PromotionKey|Quantity|UnitPrice|DiscountAmount|SalesAmount|NetSales|
+--------+------------+-----------+----------+--------+-----------+------------+--------+---------+--------------+-----------+--------+
|1       |20251212    |25400      |3840      |93      |117        |NULL        |1       |1757.28  |0.0           |1757.28    |1757.28 |
|2       |20250323    |49050      |2266      |63      |456        |NULL        |1       |7118.22  |0.0           |7118.22    |7118.22 |
|3       |20230117    |24611      |813       |17      |387        |NULL        |1       |2726.88  |0.0           |2726.88    |2726.88 |
|4       |20221201    |38217      |1813      |37      |395        |68          |5       |18402.5  |23003.12      |92012.5    |69009.38|
|5       |2

## Inspect FactSales

### Objective

Inspect the Bronze `FactSales` table before applying validations and business transformations.

The inspection includes:

- Total records
- Schema
- Sample data

In [37]:
# ==========================================================
# Inspect FactSales
# ==========================================================

print("Total Records :", fact_sales_df.count())

print("\nSchema")
fact_sales_df.printSchema()

print("\nSample Data")
fact_sales_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 44, Finished, Available, Finished, False)

Total Records : 1000000

Schema
root
 |-- SalesKey: long (nullable = true)
 |-- SalesDateKey: long (nullable = true)
 |-- CustomerKey: long (nullable = true)
 |-- ProductKey: long (nullable = true)
 |-- StoreKey: long (nullable = true)
 |-- EmployeeKey: long (nullable = true)
 |-- PromotionKey: long (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- DiscountAmount: double (nullable = true)
 |-- SalesAmount: double (nullable = true)
 |-- NetSales: double (nullable = true)


Sample Data
+--------+------------+-----------+----------+--------+-----------+------------+--------+---------+--------------+-----------+--------+
|SalesKey|SalesDateKey|CustomerKey|ProductKey|StoreKey|EmployeeKey|PromotionKey|Quantity|UnitPrice|DiscountAmount|SalesAmount|NetSales|
+--------+------------+-----------+----------+--------+-----------+------------+--------+---------+--------------+-----------+--------+
|1       |20251212    |25400      |3840      |93   

## Data Quality Validation - FactSales

### Objective

Validate the quality of the FactSales data before applying business transformations.

The following checks are performed:

- Row Count
- Null Values
- Duplicate Primary Keys
- Invalid Numeric Values

In [38]:
# ==========================================================
# Row Count
# ==========================================================

print(f"Total Records : {fact_sales_df.count()}")

# ==========================================================
# Null Validation
# ==========================================================

print("\nNull Values")

fact_sales_df.select(

    [
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)

        for c in fact_sales_df.columns
    ]

).show()

# ==========================================================
# Duplicate SalesKey Validation
# ==========================================================

duplicate_saleskey = (

    fact_sales_df

    .groupBy("SalesKey")

    .count()

    .filter(F.col("count") > 1)

)

print(f"\nDuplicate SalesKey : {duplicate_saleskey.count()}")

# ==========================================================
# Invalid Quantity
# ==========================================================

invalid_quantity = (

    fact_sales_df

    .filter(F.col("Quantity") <= 0)

)

print(f"Invalid Quantity : {invalid_quantity.count()}")

# ==========================================================
# Invalid Unit Price
# ==========================================================

invalid_price = (

    fact_sales_df

    .filter(F.col("UnitPrice") <= 0)

)

print(f"Invalid UnitPrice : {invalid_price.count()}")

# ==========================================================
# Invalid Net Sales
# ==========================================================

invalid_netsales = (

    fact_sales_df

    .filter(F.col("NetSales") < 0)

)

print(f"Negative NetSales : {invalid_netsales.count()}")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 45, Finished, Available, Finished, False)

Total Records : 1000000

Null Values
+--------+------------+-----------+----------+--------+-----------+------------+--------+---------+--------------+-----------+--------+
|SalesKey|SalesDateKey|CustomerKey|ProductKey|StoreKey|EmployeeKey|PromotionKey|Quantity|UnitPrice|DiscountAmount|SalesAmount|NetSales|
+--------+------------+-----------+----------+--------+-----------+------------+--------+---------+--------------+-----------+--------+
|       0|           0|          0|         0|       0|          0|      865088|       0|        0|             0|          0|       0|
+--------+------------+-----------+----------+--------+-----------+------------+--------+---------+--------------+-----------+--------+


Duplicate SalesKey : 0
Invalid Quantity : 0
Invalid UnitPrice : 0
Negative NetSales : 0


## Transform FactSales

### Objective

Apply business transformations to the `FactSales` table before loading it into the Silver Lakehouse.

### Transformations Performed

- Convert `PromotionKey` from Double to Long.
- Preserve nullable promotion references.
- Remove duplicate transactions.
- Preserve all business measures.

In [39]:
# ==========================================================
# Transform FactSales
# ==========================================================

dim_fact_sales_silver = (

    fact_sales_df

    # Convert PromotionKey to Long
    .withColumn(
        "PromotionKey",
        F.col("PromotionKey").cast("long")
    )

    # Remove duplicate rows
    .dropDuplicates()

)

print("FactSales transformed successfully.")

dim_fact_sales_silver.printSchema()

dim_fact_sales_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 46, Finished, Available, Finished, False)

FactSales transformed successfully.
root
 |-- SalesKey: long (nullable = true)
 |-- SalesDateKey: long (nullable = true)
 |-- CustomerKey: long (nullable = true)
 |-- ProductKey: long (nullable = true)
 |-- StoreKey: long (nullable = true)
 |-- EmployeeKey: long (nullable = true)
 |-- PromotionKey: long (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- DiscountAmount: double (nullable = true)
 |-- SalesAmount: double (nullable = true)
 |-- NetSales: double (nullable = true)

+--------+------------+-----------+----------+--------+-----------+------------+--------+---------+--------------+-----------+--------+
|SalesKey|SalesDateKey|CustomerKey|ProductKey|StoreKey|EmployeeKey|PromotionKey|Quantity|UnitPrice|DiscountAmount|SalesAmount|NetSales|
+--------+------------+-----------+----------+--------+-----------+------------+--------+---------+--------------+-----------+--------+
|1965    |20230716    |4528       |4577      |40      |82   

## Write FactSales to Silver

### Objective

Write the transformed `FactSales` table into the Silver Lakehouse as a managed Delta table.

The existing table, if present, will be overwritten.

In [40]:
# ==========================================================
# Write FactSales to Silver
# ==========================================================

(
    dim_fact_sales_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("FactSales")
)

print("FactSales successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 47, Finished, Available, Finished, False)

FactSales successfully written to LH_Silver.


## Read FactInventory

### Objective

Read the `FactInventory` table from the Bronze Lakehouse.

This table contains inventory information that will be validated and transformed before loading into the Silver Lakehouse.

In [41]:
# ==========================================================
# Read FactInventory from Bronze
# ==========================================================

fact_inventory_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/factinventory")
)

print("Bronze FactInventory Loaded Successfully")

fact_inventory_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 48, Finished, Available, Finished, False)

Bronze FactInventory Loaded Successfully
+------------+--------+----------+--------+------------+-------------+---------+------------+
|InventoryKey|DateKey |ProductKey|StoreKey|OpeningStock|StockReceived|StockSold|ClosingStock|
+------------+--------+----------+--------+------------+-------------+---------+------------+
|1           |20241216|1019      |41      |137         |53           |0        |190         |
|2           |20261223|3183      |87      |126         |62           |0        |188         |
|3           |20221020|1431      |51      |78          |45           |0        |123         |
|4           |20240129|2696      |85      |403         |15           |0        |418         |
|5           |20260713|42        |33      |145         |29           |3        |171         |
+------------+--------+----------+--------+------------+-------------+---------+------------+
only showing top 5 rows



## Inspect FactInventory

### Objective

Inspect the Bronze `FactInventory` table before applying validations and business transformations.

The inspection includes:

- Total records
- Schema
- Sample data

In [42]:
# ==========================================================
# Inspect FactInventory
# ==========================================================

print("Total Records :", fact_inventory_df.count())

print("\nSchema")
fact_inventory_df.printSchema()

print("\nSample Data")
fact_inventory_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 49, Finished, Available, Finished, False)

Total Records : 250000

Schema
root
 |-- InventoryKey: long (nullable = true)
 |-- DateKey: long (nullable = true)
 |-- ProductKey: long (nullable = true)
 |-- StoreKey: long (nullable = true)
 |-- OpeningStock: long (nullable = true)
 |-- StockReceived: long (nullable = true)
 |-- StockSold: long (nullable = true)
 |-- ClosingStock: long (nullable = true)


Sample Data
+------------+--------+----------+--------+------------+-------------+---------+------------+
|InventoryKey|DateKey |ProductKey|StoreKey|OpeningStock|StockReceived|StockSold|ClosingStock|
+------------+--------+----------+--------+------------+-------------+---------+------------+
|1           |20241216|1019      |41      |137         |53           |0        |190         |
|2           |20261223|3183      |87      |126         |62           |0        |188         |
|3           |20221020|1431      |51      |78          |45           |0        |123         |
|4           |20240129|2696      |85      |403         |15     

## Data Quality Validation - FactInventory

### Objective

Validate the quality of the FactInventory data before applying business transformations.

The following checks are performed:

- Row Count
- Null Values
- Duplicate Inventory Keys
- Invalid Stock Values

In [43]:
# ==========================================================
# Row Count
# ==========================================================

print(f"Total Records : {fact_inventory_df.count()}")

# ==========================================================
# Null Validation
# ==========================================================

print("\nNull Values")

fact_inventory_df.select(

    [
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)

        for c in fact_inventory_df.columns
    ]

).show()

# ==========================================================
# Duplicate InventoryKey
# ==========================================================

duplicate_inventorykey = (

    fact_inventory_df

    .groupBy("InventoryKey")

    .count()

    .filter(F.col("count") > 1)

)

print(f"\nDuplicate InventoryKey : {duplicate_inventorykey.count()}")

# ==========================================================
# Invalid Opening Stock
# ==========================================================

print(
    "Invalid OpeningStock :",
    fact_inventory_df.filter(F.col("OpeningStock") < 0).count()
)

# ==========================================================
# Invalid Stock Received
# ==========================================================

print(
    "Invalid StockReceived :",
    fact_inventory_df.filter(F.col("StockReceived") < 0).count()
)

# ==========================================================
# Invalid Stock Sold
# ==========================================================

print(
    "Invalid StockSold :",
    fact_inventory_df.filter(F.col("StockSold") < 0).count()
)

# ==========================================================
# Invalid Closing Stock
# ==========================================================

print(
    "Invalid ClosingStock :",
    fact_inventory_df.filter(F.col("ClosingStock") < 0).count()
)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 50, Finished, Available, Finished, False)

Total Records : 250000

Null Values
+------------+-------+----------+--------+------------+-------------+---------+------------+
|InventoryKey|DateKey|ProductKey|StoreKey|OpeningStock|StockReceived|StockSold|ClosingStock|
+------------+-------+----------+--------+------------+-------------+---------+------------+
|           0|      0|         0|       0|           0|            0|        0|           0|
+------------+-------+----------+--------+------------+-------------+---------+------------+


Duplicate InventoryKey : 0
Invalid OpeningStock : 0
Invalid StockReceived : 0
Invalid StockSold : 0
Invalid ClosingStock : 0


## Transform FactInventory

### Objective

Apply business transformations to the `FactInventory` table.

### Transformations Performed

- Remove duplicate records.
- Preserve inventory measures.
- Preserve all foreign keys.

In [44]:
# ==========================================================
# Transform FactInventory
# ==========================================================

fact_inventory_silver = (

    fact_inventory_df

    # Remove duplicate rows
    .dropDuplicates()

)

print("FactInventory transformed successfully.")

fact_inventory_silver.printSchema()

fact_inventory_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 51, Finished, Available, Finished, False)

FactInventory transformed successfully.
root
 |-- InventoryKey: long (nullable = true)
 |-- DateKey: long (nullable = true)
 |-- ProductKey: long (nullable = true)
 |-- StoreKey: long (nullable = true)
 |-- OpeningStock: long (nullable = true)
 |-- StockReceived: long (nullable = true)
 |-- StockSold: long (nullable = true)
 |-- ClosingStock: long (nullable = true)

+------------+--------+----------+--------+------------+-------------+---------+------------+
|InventoryKey|DateKey |ProductKey|StoreKey|OpeningStock|StockReceived|StockSold|ClosingStock|
+------------+--------+----------+--------+------------+-------------+---------+------------+
|255         |20220513|1438      |51      |308         |44           |0        |352         |
|326         |20230527|4642      |73      |290         |29           |0        |319         |
|336         |20260917|728       |24      |81          |16           |0        |97          |
|343         |20221002|2309      |68      |86          |20         

## Write FactInventory to Silver

### Objective

Write the transformed `FactInventory` table into the Silver Lakehouse as a managed Delta table.

If the table already exists, it will be overwritten.

In [45]:
# ==========================================================
# Write FactInventory to Silver
# ==========================================================

(
    fact_inventory_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("FactInventory")
)

print("FactInventory successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 52, Finished, Available, Finished, False)

FactInventory successfully written to LH_Silver.


## Read FactReturns

### Objective

Read the `FactReturns` table from the Bronze Lakehouse.

This table contains returned product transactions and will be validated and transformed before loading into the Silver Lakehouse.

In [46]:
# ==========================================================
# Read FactReturns from Bronze
# ==========================================================

fact_returns_df = (
    spark.read
         .format("delta")
         .load(f"{bronze_base_path}/factreturns")
)

print("Bronze FactReturns Loaded Successfully")

fact_returns_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 53, Finished, Available, Finished, False)

Bronze FactReturns Loaded Successfully
+---------+--------+-------------+-----------+----------+--------+--------------+------------+---------------------+
|ReturnKey|SalesKey|ReturnDateKey|CustomerKey|ProductKey|StoreKey|ReturnQuantity|ReturnAmount|ReturnReason         |
+---------+--------+-------------+-----------+----------+--------+--------------+------------+---------------------+
|1        |3       |20230129     |24611      |813       |17      |1             |2726.88     |Customer Changed Mind|
|2        |4       |20221212     |38217      |1813      |37      |3             |41405.63    |Quality Issue        |
|3        |19      |20230806     |45277      |467       |50      |1             |2738.12     |Quality Issue        |
|4        |25      |20270712     |37124      |874       |37      |1             |4630.97     |Late Delivery        |
|5        |57      |20220310     |27729      |1843      |14      |1             |3232.67     |Late Delivery        |
+---------+--------+-----

## Inspect FactReturns

### Objective

Inspect the Bronze `FactReturns` table before applying validations and business transformations.

The inspection includes:

- Total records
- Schema
- Sample data

In [47]:
# ==========================================================
# Inspect FactReturns
# ==========================================================

print("Total Records :", fact_returns_df.count())

print("\nSchema")
fact_returns_df.printSchema()

print("\nSample Data")
fact_returns_df.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 54, Finished, Available, Finished, False)

Total Records : 77065

Schema
root
 |-- ReturnKey: long (nullable = true)
 |-- SalesKey: long (nullable = true)
 |-- ReturnDateKey: long (nullable = true)
 |-- CustomerKey: long (nullable = true)
 |-- ProductKey: long (nullable = true)
 |-- StoreKey: long (nullable = true)
 |-- ReturnQuantity: long (nullable = true)
 |-- ReturnAmount: double (nullable = true)
 |-- ReturnReason: string (nullable = true)


Sample Data
+---------+--------+-------------+-----------+----------+--------+--------------+------------+---------------------+
|ReturnKey|SalesKey|ReturnDateKey|CustomerKey|ProductKey|StoreKey|ReturnQuantity|ReturnAmount|ReturnReason         |
+---------+--------+-------------+-----------+----------+--------+--------------+------------+---------------------+
|1        |3       |20230129     |24611      |813       |17      |1             |2726.88     |Customer Changed Mind|
|2        |4       |20221212     |38217      |1813      |37      |3             |41405.63    |Quality Issue     

## Data Quality Validation - FactReturns

### Objective

Validate the quality of the FactReturns data before applying business transformations.

The following checks are performed:

- Row Count
- Null Values
- Duplicate Return Keys
- Invalid Return Quantity
- Invalid Return Amount

In [48]:
# ==========================================================
# Row Count
# ==========================================================

print(f"Total Records : {fact_returns_df.count()}")

# ==========================================================
# Null Validation
# ==========================================================

print("\nNull Values")

fact_returns_df.select(

    [
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)

        for c in fact_returns_df.columns
    ]

).show()

# ==========================================================
# Duplicate ReturnKey
# ==========================================================

duplicate_returnkey = (

    fact_returns_df

    .groupBy("ReturnKey")

    .count()

    .filter(F.col("count") > 1)

)

print(f"\nDuplicate ReturnKey : {duplicate_returnkey.count()}")

# ==========================================================
# Invalid Return Quantity
# ==========================================================

print(
    "Invalid ReturnQuantity :",
    fact_returns_df.filter(F.col("ReturnQuantity") <= 0).count()
)

# ==========================================================
# Invalid Return Amount
# ==========================================================

print(
    "Invalid ReturnAmount :",
    fact_returns_df.filter(F.col("ReturnAmount") < 0).count()
)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 55, Finished, Available, Finished, False)

Total Records : 77065

Null Values
+---------+--------+-------------+-----------+----------+--------+--------------+------------+------------+
|ReturnKey|SalesKey|ReturnDateKey|CustomerKey|ProductKey|StoreKey|ReturnQuantity|ReturnAmount|ReturnReason|
+---------+--------+-------------+-----------+----------+--------+--------------+------------+------------+
|        0|       0|            0|          0|         0|       0|             0|           0|           0|
+---------+--------+-------------+-----------+----------+--------+--------------+------------+------------+


Duplicate ReturnKey : 0
Invalid ReturnQuantity : 0
Invalid ReturnAmount : 0


## Transform FactReturns

### Objective

Apply business transformations to the `FactReturns` table.

### Transformations Performed

- Standardize return reason values.
- Remove duplicate records.
- Preserve business keys and measures.

In [49]:
# ==========================================================
# Transform FactReturns
# ==========================================================

fact_returns_silver = (

    fact_returns_df

    # Return Reason
    .withColumn(
        "ReturnReason",
        F.initcap(F.trim(F.col("ReturnReason")))
    )

    # Remove duplicate rows
    .dropDuplicates()

)

print("FactReturns transformed successfully.")

fact_returns_silver.printSchema()

fact_returns_silver.show(5, truncate=False)

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 56, Finished, Available, Finished, False)

FactReturns transformed successfully.
root
 |-- ReturnKey: long (nullable = true)
 |-- SalesKey: long (nullable = true)
 |-- ReturnDateKey: long (nullable = true)
 |-- CustomerKey: long (nullable = true)
 |-- ProductKey: long (nullable = true)
 |-- StoreKey: long (nullable = true)
 |-- ReturnQuantity: long (nullable = true)
 |-- ReturnAmount: double (nullable = true)
 |-- ReturnReason: string (nullable = true)

+---------+--------+-------------+-----------+----------+--------+--------------+------------+---------------------+
|ReturnKey|SalesKey|ReturnDateKey|CustomerKey|ProductKey|StoreKey|ReturnQuantity|ReturnAmount|ReturnReason         |
+---------+--------+-------------+-----------+----------+--------+--------------+------------+---------------------+
|367      |5117    |20261113     |13683      |3289      |67      |1             |6372.09     |Late Delivery        |
|411      |5794    |20270712     |8980       |3479      |31      |3             |7244.07     |Quality Issue        |


## Write FactReturns to Silver

### Objective

Write the transformed `FactReturns` table into the Silver Lakehouse as a managed Delta table.

If the table already exists, it will be overwritten.

In [50]:
# ==========================================================
# Write FactReturns to Silver
# ==========================================================

(
    fact_returns_silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("FactReturns")
)

print("FactReturns successfully written to LH_Silver.")

StatementMeta(, ee53fac9-6937-41a2-8f78-43af4e2bad3f, 57, Finished, Available, Finished, False)

FactReturns successfully written to LH_Silver.
